## Ćwiczenie 7: Wizualizacja działania operacji morfologicznych

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML, display

# 1. Obraz wejściowy (litera 'C' + pojedynczy piksel szumu)
image = np.zeros((10, 10), dtype=int)
image[2:8, 3] = 1
image[2, 3:6] = 1
image[7, 3:6] = 1
image[5, 8] = 1

# 2. Element strukturalny w kształcie krzyża
selem = np.array([
    [0, 1, 0],
    [1, 1, 1],
    [0, 1, 0]
])
cross_offsets = [(0, 0), (-1, 0), (1, 0), (0, -1), (0, 1)]

# 3. Symulacja dylatacji
pad_img = np.pad(image, pad_width=1, mode='constant', constant_values=0)
dilated_result = np.zeros_like(image)
frames_dilation = []

for r in range(image.shape[0]):
    for c in range(image.shape[1]):
        region = pad_img[r:r+3, c:c+3]
        if np.any(np.logical_and(region, selem)):
            dilated_result[r, c] = 1
        frames_dilation.append((r, c, dilated_result.copy()))

# --- Animacja Dylatacji ---
fig_d, (ax1_d, ax2_d) = plt.subplots(1, 2, figsize=(10, 5))
im1_d = ax1_d.imshow(image, cmap='gray', vmin=0, vmax=1)
ax1_d.set_title("Obraz Wejściowy (Z czerwonym krzyżem)")
ax1_d.set_xticks(range(10)); ax1_d.set_yticks(range(10))

im2_d = ax2_d.imshow(np.zeros_like(image), cmap='gray', vmin=0, vmax=1)
title_d = ax2_d.set_title("Dylatacja: Szukanie jakiejkolwiek '1'")
ax2_d.set_xticks(range(10)); ax2_d.set_yticks(range(10))

patches_d = [plt.Rectangle((0, 0), 1, 1, fill=False, edgecolor='red', lw=2) for _ in range(5)]
for p in patches_d:
    ax1_d.add_patch(p)

def update_dilation(frame_data):
    r, c, current_result = frame_data
    for patch, (dr, dc) in zip(patches_d, cross_offsets):
        patch.set_xy((c + dc - 0.5, r + dr - 0.5))
    im2_d.set_data(current_result)
    title_d.set_text(f"Dylatacja: Wiersz {r}, Kolumna {c}")
    return [im1_d, im2_d] + patches_d

ani_dilation = animation.FuncAnimation(
    fig_d, update_dilation, frames=frames_dilation, interval=100, repeat=False
)
plt.close(fig_d)
HTML(ani_dilation.to_jshtml())

### Jak czytać animację Dylatacji?
Czerwony "celownik" w kształcie krzyża nakładany jest na obraz wejściowy. Jeśli pod dowolnym z 5 pól krzyża znajduje się biały piksel (`1`), centralny piksel obrazu wynikowego staje się biały.

In [5]:
# --- Animacja Erozji ---
eroded_result = np.zeros_like(image)
frames_erosion = []
selem_sum = np.sum(selem)

for r in range(image.shape[0]):
    for c in range(image.shape[1]):
        region = pad_img[r:r+3, c:c+3]
        if np.sum(np.logical_and(region, selem)) == selem_sum:
            eroded_result[r, c] = 1
        frames_erosion.append((r, c, eroded_result.copy()))

fig_e, (ax1_e, ax2_e) = plt.subplots(1, 2, figsize=(10, 5))
im1_e = ax1_e.imshow(image, cmap='gray', vmin=0, vmax=1)
ax1_e.set_title("Obraz Wejściowy (Z czerwonym krzyżem)")
ax1_e.set_xticks(range(10)); ax1_e.set_yticks(range(10))

im2_e = ax2_e.imshow(np.zeros_like(image), cmap='gray', vmin=0, vmax=1)
title_e = ax2_e.set_title("Erozja: Dopasowanie idealne (Wszystkie 5)")
ax2_e.set_xticks(range(10)); ax2_e.set_yticks(range(10))

patches_e = [plt.Rectangle((0, 0), 1, 1, fill=False, edgecolor='red', lw=2) for _ in range(5)]
for p in patches_e:
    ax1_e.add_patch(p)

def update_erosion(frame_data):
    r, c, current_result = frame_data
    for patch, (dr, dc) in zip(patches_e, cross_offsets):
        patch.set_xy((c + dc - 0.5, r + dr - 0.5))
    im2_e.set_data(current_result)
    title_e.set_text(f"Erozja: Wiersz {r}, Kolumna {c}")
    return [im1_e, im2_e] + patches_e

ani_erosion = animation.FuncAnimation(
    fig_e, update_erosion, frames=frames_erosion, interval=100, repeat=False
)
plt.close(fig_e)
HTML(ani_erosion.to_jshtml())

### Jak czytać animację Erozji?
Aby centralny piksel w obrazie wynikowym pozostał biały (`1`), pod wszystkimi 5 ramionami czerwonego krzyża jednocześnie muszą znajdować się białe piksele (`1`). Jeśli choć jedno ramię wychodzi poza obszar lub trafia w czarne tło, wynik staje się czarny (`0`).